# **1. Considerations when Deploying Platforms**

> Training a model is one chapter. **Deploying and maintaining it** is the rest of the book.

---

## **1.1 Preparing Model for Deployment**

**Goal:** make a trained model **smaller, faster and cheaper** to run.
This step is **optional**. Do it only when the model is too slow, too big or too costly.

**Analogy: a huge textbook you need to carry around**
* **Pruning** = tear out the pages nobody reads
* **Quantization** = print the same pages in lighter ink so the book weighs less
* **Distillation** = an expert writes a short summary book teaching the same material

### **A. Pruning**
**Remove weights that barely affect the output.**

```
Before:  w = [0.82, 0.001, -0.67, 0.0003, 0.45]
After:   w = [0.82,   0,   -0.67,   0,    0.45]
```
* Usually followed by a **brief fine tune** to recover accuracy

### **B. Quantization**
**Store weights with fewer bits.**

```
float32:  0.8234719   (32 bits)
int8:     105         (8 bits + a scale factor)
```
* About **4x less memory** at 8 bit
* **Faster math** on supported hardware
* Small precision loss, most models barely notice
* Often needs **no retraining** (done after training)
* **Cheapest optimization to try first**

### **C. Distillation**
**Train a small "student" model to imitate a large "teacher" model.**

```
Hard label:      cat = 1,    dog = 0,    fox = 0
Teacher output:  cat = 0.85, dog = 0.10, fox = 0.05
```
* The student learns the teacher's **full probability output**, which carries richer information
* A distilled student often beats a same size model trained from scratch

### **Comparison**

| | What changes | Needs retraining? |
|:---|:---|:---|
| **Pruning** | Removes weights | Usually a brief fine tune |
| **Quantization** | Shrinks number precision | Often none |
| **Distillation** | Creates a new smaller model | Yes, trains the student |

**They stack:** distill first, then prune, then quantize.

---

## **1.2 Model Versioning**

**Store every model build with a label and its context.**

```
atas-classifier
 ├── v1  → MobileNetV2, 1000 image subset, acc 61%
 ├── v2  → EfficientNetV2-L, full data, acc 78.08%   ← production
 └── v3  → fine tuned, new classes, acc ???          ← testing
```

**Each version records:**
* The **model file**
* The **training data** used
* The **code and hyperparameters**
* The **metrics**

**What it gives you:**
* **Rollback:** v3 misbehaves, switch back to v2 in minutes
* **Reproducibility:** explain exactly how any model was built
* **Safe rollout:** send 10% of traffic to v3, compare, then promote

**Where it lives:** **Azure ML Model Registry** and **SageMaker Model Registry**

---
## **1.3 CI/CD and Pipelines**

**Analogy: a robot that does your deploying steps for you, every time.**

Without CI/CD, after every code change you manually test, build the Docker image and upload it. Slow, and you will eventually forget a step.

```
YOU                          ROBOT (runs on a server)
────                         ────────────────────────
git push  ───────────────►   1. Download your code
                             2. Run your tests
                             3. Tests fail → STOP, notify you
                             4. Tests pass → build Docker image
                             5. Deploy it
```

### **Key terms**
* **CI (Continuous Integration):** steps 1 to 3. **Automatically check that the code still works** on every push.
* **CD (Continuous Delivery / Deployment):** steps 4 and 5. **Automatically ship it** if the checks pass.
* **Pipeline:** the whole list of steps. A **recipe written once in a file**, run by the robot on every push.

```
Continuous DELIVERY    → robot prepares everything, a human presses "Deploy"
Continuous DEPLOYMENT  → robot deploys with no human involved
```

### **`git push` is NOT CI/CD**
**`git push` is the trigger. The pipeline is what runs after it.**

```
git push  →  only uploads code to GitHub, nothing else happens
CI/CD     →  what runs AFTER the push: test → build → deploy
```
**Doorbell analogy:** git push = pressing the bell. Pipeline = everything that happens inside the house.

### **Who writes what**
```
You write:       test_model.py
                 "assert accuracy >= 0.75"
                 "assert API returns a result for a sample image"

Pipeline does:   runs the tests on every push
                 pass → continue to deploy
                 fail → stop, nothing goes live
```
**Your tests are the rules. The pipeline is the gatekeeper that enforces them.**

A pipeline can also build, deploy and notify, not only test.

### **The ML twist**
In ML the **model is also an artifact**, so extra stages appear:
```
Push code → tests → retrain → evaluate vs current model
          → if better, register new version → deploy
```
**Typical gate:** the new model must score **at least as well as the current one**, otherwise do not deploy.

**Tools:** GitHub Actions, Azure DevOps, Jenkins, GitLab CI

## **1.4 Load Testing**

**Analogy:** invite 200 people to a restaurant at once before opening day.

**Simulate many users at the same time to find the breaking point.**
```
10 users    → latency 0.4s   OK
100 users   → latency 1.2s   OK
1000 users  → latency 9s, errors appear   ← breaking point
```

| Metric | Meaning |
|:---|:---|
| **Latency** | Time per request |
| **Throughput** | Requests handled per second |
| **Error rate** | Percent of failed requests |
| **Resource use** | CPU, memory, GPU |

* **Stress testing:** push past the limit to see how it fails
* **Spike testing:** sudden jump in traffic
* **Tools:** Locust (Python), k6, Apache JMeter
* Usually runs as a **stage inside the CD pipeline, on staging**

---

## **1.5 Model Monitoring**

**Analogy: watching the kitchen while the restaurant is open.**

CI/CD checks the model **before** it goes live. **Monitoring watches it after.**

**The core problem:** a normal app crashes and you see the error. A model can **get worse with no error at all.** It keeps answering, just wrongly more often.

```
Training time:    clean, well lit aircraft photos           → 78% accuracy
Six months later: blurry phone photos, night shots,
                  new aircraft types                        → maybe 55%

Server status: running fine. No errors. No alerts.
Reality: the model is quietly failing.
```
**Nothing in the code changed. The world changed and the model did not.**

### **Three things to monitor**
```
1. Operational health  → Is it fast? Up? Any errors? Cost?
2. Data drift          → Do inputs still look like the training data?
3. Model performance   → Is accuracy still acceptable?
```

### **Drift types**
```
Data drift:     the INPUTS change
                (trained on clean photos, now receiving blurry ones)

Concept drift:  the RULE linking input to answer changes
                (fraud tricks evolve, so "fraud" looks different over time)
```

### **How it works**
```
Model is live
    ↓
Log every input and prediction
    ↓
Compare against a baseline (usually the training data)
    ↓
Diverge too much → send an alert
```

**Catch:** checking accuracy needs the correct answers, which you often **do not have right away** in production. That is why **input drift checks are popular**: they need no labels and warn you early.

### **Cloud tools**

| | Azure | AWS |
|:---|:---|:---|
| **ML platform** | Azure Machine Learning | Amazon SageMaker |
| **Drift and quality checks** | Azure ML model monitoring | SageMaker Model Monitor |
| **Infra metrics, logs, alerts** | Azure Monitor, Application Insights | Amazon CloudWatch |

---

## **1.6 The Full Lifecycle**

```
BEFORE DEPLOY   Pruning / Quantization / Distillation   (optional)

SHIPPING        Versioning → CI/CD pipeline → Load testing (on staging)

SERVING         Expose the model as an API (example: FastAPI + Docker)

LIVE            Monitoring
```

### **The maintenance loop**
```
        ┌─────────────────────────────────────────┐
        ↓                                         │
   Train model                                    │
        ↓                                         │
   Version it (v1, v2, v3...)                     │
        ↓                                         │
   CI/CD pipeline: test → build → deploy          │
        ↓                                         │
   Model is LIVE                                  │
        ↓                                         │
   Monitoring watches it                          │
        ↓                                         │
   Drift or accuracy drop detected ───────────────┘
   (retrain with fresh data, register as next version)
```

### **One line each**
```
CI/CD       → the gate BEFORE going live
Monitoring  → the watchman AFTER going live
Versioning  → the filing cabinet, so you can roll back
Retraining  → what you do when the watchman raises an alarm
```

### **Example walk through (ATAS)**
```
1. v2 is live, 78% accuracy
2. Monitoring notices average confidence falling over weeks
3. Collect new blurry and night images, retrain → v3
4. Push v3 → pipeline checks "v3 must beat v2 on the test set"
5. v3 passes → deployed, v2 kept in the registry
6. v3 misbehaves → roll back to v2 in minutes
```

**Key takeaways**
* **Monitoring detects** the problem, **versioning provides** the fix
* **Optimization is optional**, the rest forms the core loop
* **Versioning and CI/CD** are set up early and run on every change
* **Monitoring starts at go live and never stops**
---

## **1.7 Packaging and Containerizing Models**

### **The problem**
The model works on my laptop but breaks elsewhere: different Python version,
different library versions, missing preprocessing file.
Analogy: a cloud kitchen ships the dish sealed in a box with everything needed.
The container is that box.

### **Two separate ideas**
```
PACKAGING                      CONTAINERIZING
what the model needs to run    where it runs, identical everywhere
```

### **Four benefits**
* Portability: runs the same on any machine
* Scalability: start many copies of the same image
* Reproducibility: same behavior in dev, test, prod
* Isolation: dependencies cannot clash with other apps

### **What a package contains**
```
trained model file (weights, .pkl, .keras)
+ preprocessing code (scaler, resizing)
+ postprocessing code (thresholds, recommendation rules)
+ requirements.txt (pinned versions)
+ metadata (name, version)
```
Dependencies are what the model needs to RUN (inference), not to learn.
If the scaler is not shipped with the model, predictions go silently wrong.

### **Dockerfile, Image, Container**
```
Dockerfile  the recipe, the steps in order
     ↓ docker build
Image       the frozen blueprint (OS layer, Python, libraries, code)
     ↓ docker run
Container   a live running copy of the image
```
Image is like a class, container is like an object.
One image can start many containers.
Deleting a container does not delete the image.

```
docker images   lists images
docker ps       lists running containers
docker logs <name>   shows what the app printed (first place to debug)
```

### **Step by step flow**
Phase A: package
1. Save the trained model (joblib.dump, model.save)
2. Pin dependencies in requirements.txt with exact versions
3. Include preprocessing and postprocessing as functions

```
raw input → preprocess() → model → postprocess() → answer
```

Phase B: containerize
4. Write the Dockerfile
5. Build the image
6. Run the container
7. Test it locally

### **Pinning dependencies**
```
conda env (trained here) → pip freeze → requirements.txt → docker build
```
* pip freeze only PRINTS installed versions. Saving them into requirements.txt
  is what pins them. The image is what freezes them.
* Unpinned versions drift over time. Failures are either loud (model will not
  load) or silent (predictions quietly change). Silent is more dangerous.
* Trim the freeze output to what inference needs, to keep the image small.

### **Dockerfile line by line**
```
FROM python:3.8-slim              base image with Linux and Python
WORKDIR /app                      working folder inside the image
COPY . /app                       copy project files into the image
RUN pip install -r requirements.txt   install libraries (build time, once)
EXPOSE 80                         documents the port (does not open it)
CMD ["python", "app.py"]          start the app (run time, every start)
```
RUN happens once at build time. CMD happens every time the container starts.
Changing requirements.txt means rebuilding the image. Running containers keep
the old image until replaced.
On Hugging Face Spaces the port is 7860.

### **Testing locally**
```
docker run → container live → send a request → check the answer
```
1. Did it start? (docker ps, docker logs)
2. Does the endpoint respond? (curl to /predict)
3. Is the answer correct and in the right format?
Works in the conda env does not mean works in the container.

### **Five best practices**
1. Lightweight base images: use slim, smaller and faster, fewer attack points
2. Minimize layers: combine related commands, each instruction is a layer
3. Secrets at run time, never baked in: images get shared, and a key stays in
   the layers even if deleted later. A leaked key must be revoked.
4. Update images regularly: base image and libraries collect vulnerabilities.
   Update deliberately: update, rebuild, test, deploy.
5. Automate builds with CI: git push triggers a pipeline that builds, tests,
   and pushes only if tests pass. Scheduled rebuilds keep images patched.

### **One line summary**
The image freezes the pinned libraries from requirements.txt, and each
container started from it behaves the same, because the environment ships
with the model.

### **ATAS mapping**
```
model.pkl / .h5       EfficientNetV2 weights + 2 XGBoost models
requirements.txt      pinned libraries
preprocess/postprocess image resizing, feature building, recommendation rules
Flask /predict        FastAPI /analyze
Dockerfile            Dockerfile for the Hugging Face Space
```